In [6]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path(os.getcwd()).resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from torchsummary import summary
from PIL import Image

import torch
import torch.nn.functional as F                          # Functions like ReLU
import torch.optim as optim                              # Optimizers like Adam
import torch.nn as nn                                    # Neural Network Modules


# from load_data import CatDogDataLoadandSave, CatandDogDataLoader
# from vgg16_model import VGG16

In [14]:
!git clone https://github.com/gimoonnam/vgg16_practice.git

Cloning into 'vgg16_practice'...
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 0), reused 5 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (5/5), 5.60 KiB | 5.60 MiB/s, done.


In [24]:
sys.path.insert(0, '/content/vgg16_practice')

In [32]:
sys.path

['/content/vgg16_practice',
 '/content/vgg16_practice',
 '/content/vgg16_practice',
 '/content/vgg16_practice',
 '/content',
 '/env/python',
 '/usr/lib/python312.zip',
 '/usr/lib/python3.12',
 '/usr/lib/python3.12/lib-dynload',
 '',
 '/usr/local/lib/python3.12/dist-packages',
 '/usr/lib/python3/dist-packages',
 '/usr/local/lib/python3.12/dist-packages/IPython/extensions',
 '/root/.ipython',
 '/content',
 '/content',
 '/content',
 '/content']

In [19]:
!ls /content/vgg16_practice

load_data.py  vgg16_from_scratch.ipynb	vgg16_model.py


In [34]:
from vgg16_model import VGG16
# from .vgg16_model import VGG16

ModuleNotFoundError: No module named 'vgg16_model'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# import os
# import pickle
# import datetime
# import wandb
# from google.colab import userdata, drive

# # set the path in google drive where toxic-comments data are placed
# drive.mount('/content/drive/')
# working_dir = '/content/drive/MyDrive/ColabNotebooks/DistilBert/Toxic_comments_classification'
# os.chdir(working_dir)



### Load data and Save dataset as ubyte format


In [1]:
data_path = r'/content/drive/MyDrive/Data'
train_data_path = os.path.join(data_path, "cat-and-dog/training_set/")
test_data_path  = os.path.join(data_path, "cat-and-dog/test_set/")

# load data
ds_train = CatDogDataLoadandSave(data_dir=train_data_path)
ds_test = CatDogDataLoadandSave(data_dir=test_data_path)

# save them as ubyte format
output_directory = os.path.join(data_path, "cat-and-dog/ubyte_format")
ds_train.save_as_ubyte(output_dir=output_directory, img_size=(224, 224), prefix="catdog_train")
ds_test.save_as_ubyte(output_dir=output_directory, img_size=(224, 224), prefix="catdog_test")

NameError: name 'os' is not defined

### create torch data loader

In [ ]:
output_directory = r"/content/drive/MyDrive/Data/cat-and-dog/ubyte_format"

ds_train = CatandDogDataLoader(raw_folder=output_directory, train=True)
train_loader = torch.utils.data.DataLoader(ds_train, batch_size=64, shuffle=True)

Loaded 8005 images of shape 224x224x3
Loaded 2049 labels


### Build VGG16 architecture

In [ ]:
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(device)

model = VGG16(3, 2).to(device)

mps


In [ ]:
learning_rate = 1e-4
num_epoches = 20

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
for epoch in range(num_epoches):
    ProgressBar = tqdm(enumerate(train_loader), total=len(train_loader))

    model.train()
    running_loss = 0.0
    for batch_idx, (inputs, labels) in ProgressBar:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()


        #Update Progress bar
        ProgressBar.set_description(f'Epoch [{epoch+1}]')
        ProgressBar.set_postfix(loss=loss.item())

Epoch [2]:  10%|█         | 13/126 [00:56<08:15,  4.38s/it, loss=0.691]


KeyboardInterrupt: 